# Model 2: Transfer Learning with VGG16 Convolutional Base for Lego Brick Classification

## Aim

The aim of this experiment is to apply transfer learning using the pre-trained VGG16 network to classify Lego brick images into five categories. In Model 2, all of the original dense layers of VGG16 are removed and replaced with a new dense output layer containing 5 neurons. The convolutional base is frozen so that only the replacement classifier is trained.

In [ ]:
import os
import zipfile
import shutil
import random
from pathlib import Path

## Dataset Extraction

The dataset is stored as a zip file and is extracted into the Colab runtime before preprocessing and splitting. The class folders are labelled 0, 1, 2, 3 and 4.

In [ ]:
zip_path = "small_Lego.zip"  
extract_path = "dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully")
print(os.listdir(extract_path))

## Check Dataset Structure

The extracted dataset is checked to confirm that it contains five class folders. These folders are used by Keras to automatically assign class labels.

In [ ]:
base_data_dir = "dataset/SmallLego"
print(os.listdir(base_data_dir))

## Train, Validation and Test Split

The dataset is divided into training, validation and test sets using a 60/20/20 ratio. The same split is used for both Model 1 and Model 2 so that the comparison between the two models is fair.

In [ ]:
random.seed(42)

source_dir = Path(base_data_dir)
split_root = Path("split_data")

train_dir = split_root / "train"
val_dir = split_root / "val"
test_dir = split_root / "test"

if split_root.exists():
    shutil.rmtree(split_root)

for split in [train_dir, val_dir, test_dir]:
    split.mkdir(parents=True, exist_ok=True)

class_names = sorted([d.name for d in source_dir.iterdir() if d.is_dir()])
print("Classes:", class_names)

for class_name in class_names:
    class_source = source_dir / class_name
    images = [f for f in class_source.iterdir() if f.is_file()]
    random.shuffle(images)

    total = len(images)
    train_count = int(total * 0.6)
    val_count = int(total * 0.2)
    test_count = total - train_count - val_count

    class_train = train_dir / class_name
    class_val = val_dir / class_name
    class_test = test_dir / class_name

    class_train.mkdir(parents=True, exist_ok=True)
    class_val.mkdir(parents=True, exist_ok=True)
    class_test.mkdir(parents=True, exist_ok=True)

    train_files = images[:train_count]
    val_files = images[train_count:train_count + val_count]
    test_files = images[train_count + val_count:]

    for f in train_files:
        shutil.copy(f, class_train / f.name)

    for f in val_files:
        shutil.copy(f, class_val / f.name)

    for f in test_files:
        shutil.copy(f, class_test / f.name)

    print(f"Class {class_name}: train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")

## Pre-processing

The images are resized to 224 × 224 × 3, which is the required input size for VGG16. Standard VGG16 preprocessing is used so that the images undergo mean subtraction based on the ImageNet training procedure. No additional rescaling to the range 0 to 1 is applied.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input

In [ ]:
batch_size = 32
img_size = (224, 224)

train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

## Model 2 Architecture

Model 2 uses the pre-trained VGG16 convolutional base only. All of the original fully connected dense layers are removed. The output of the convolutional base is flattened and connected directly to a new dense output layer of 5 neurons with softmax activation. All convolutional layers are frozen so that only the final replacement classifier is trained.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = Flatten()(x)
output = Dense(5, activation='softmax')(x)

model2 = Model(inputs=base_model.input, outputs=output)

model2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Model Summary

The summary below is used to confirm that Model 2 matches the required specification. According to the assignment, Model 2 should have approximately 14 million total parameters, with approximately 125,445 trainable parameters.

In [ ]:
model2.summary()

## Training Setup

The model is trained for 10 epochs using categorical cross-entropy loss and the Adam optimiser. Training and validation accuracy and loss are recorded after each epoch so that the learning behaviour of the model can be analysed.

In [ ]:
history2 = model2.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

## Save the Model

The trained model is saved after training so that it can be reloaded later without retraining.

In [ ]:
model2.save("model2.h5")
print("Model 2 saved!")

## Training Curves

The following graphs show training and validation accuracy and loss across 10 epochs. These plots are used to assess whether the model learned effectively and whether any overfitting occurred.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history2.history['accuracy'], label='Training Accuracy')
plt.plot(history2.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model 2 Training and Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history2.history['loss'], label='Training Loss')
plt.plot(history2.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model 2 Training and Validation Loss')
plt.legend()
plt.show()

## Interpretation of the Training Curves

The training and validation curves are used to assess model behaviour during learning. If both training and validation accuracy improve while loss decreases, the model is learning useful patterns from the data. If training performance improves substantially more than validation performance, this may indicate overfitting.

## Test Set Evaluation

After 10 epochs, the final model is evaluated on the test set. The test accuracy provides a single numerical summary of model performance on unseen data.

In [ ]:
test_loss, test_accuracy = model2.evaluate(test_data)
print("Model 2 Test Loss:", test_loss)
print("Model 2 Test Accuracy:", test_accuracy)

## Confusion Matrix

A 5 × 5 confusion matrix is used to examine how predictions are distributed across the five classes. This provides more detailed insight than overall accuracy because it shows which classes are most frequently confused.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
pred_probs = model2.predict(test_data)
pred_labels = np.argmax(pred_probs, axis=1)
true_labels = test_data.classes

cm = confusion_matrix(true_labels, pred_labels)
print(cm)

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(test_data.class_indices.keys())
)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title("Model 2 Confusion Matrix")
plt.show()

## Conclusion

Model 2 applies transfer learning by keeping only the pre-trained VGG16 convolutional base and replacing the original dense classification layers with a new 5-class output layer. This approach removes much of the original ImageNet classifier and requires the model to rely mainly on the transferred convolutional feature extractor for Lego brick classification.